# 03 — Pipeline CNBI e comparações

Implementa payoff, orientação canônica, análise paralela legada independente, janela singular, deltas adaptativos e subproblemas CNBI. O SMOKE valida o núcleo; PILOT/FULL habilitam os métodos completos sob contador real.

In [ ]:
from pathlib import Path
import json, os, time, math, gc
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize, minimize_scalar
from scipy.spatial import ConvexHull, Delaunay, cKDTree
from scipy.stats import qmc

def project_root(start=Path.cwd()):
    p=start.resolve()
    for candidate in (p,*p.parents):
        if (candidate/'configs'/'smoke.json').exists(): return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada')

ROOT=project_root(); MODE=os.environ.get('CNBI_MODE','SMOKE').upper()
CFG=json.loads((ROOT/'configs'/f'{MODE.lower()}.json').read_text(encoding='utf-8'))
for key in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMEXPR_NUM_THREADS'): os.environ[key]='1'
ALPHA=2**0.75; DELTA_BY_K={2:.10,3:.10,4:.20,5:.50}


import hashlib
SCHEMA_VERSION=2
IMPLEMENTATION_FINGERPRINT='common-nbi-valid-payoff-v4'
def stable_hash(value):
    if isinstance(value,np.ndarray): payload=np.ascontiguousarray(value).tobytes()
    else: payload=json.dumps(value,sort_keys=True,separators=(',',':')).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()
def checkpoint_identity(scenario,method,seed,B,A,budget=None,parameters=None,stage=None):
    body={'schema_version':SCHEMA_VERSION,'implementation_fingerprint':IMPLEMENTATION_FINGERPRINT,'mode':MODE,'scenario':scenario,'method':method,'dimension':int(B.shape[1]),'seed':int(seed),'budget':None if budget is None else int(budget),'parameters':parameters or {},'stage':stage,'config_hash':stable_hash(CFG),'anchors_hash':stable_hash(A),'rsm_hash':stable_hash(B)}
    body['fingerprint']=stable_hash(body); return body
def checkpoint_matches(meta,identity): return all(meta.get(k)==v for k,v in identity.items())

from itertools import combinations,product
from scipy.linalg import null_space
OUT=ROOT/'data'/'generated'; OUT=OUT/'smoke' if MODE=='SMOKE' else OUT; CK=ROOT/'results'/'synthetic'/'checkpoints'; TAB=ROOT/'results'/'synthetic'/'tables'; CK.mkdir(parents=True,exist_ok=True); TAB.mkdir(parents=True,exist_ok=True)

class EvaluationCounter:
    def __init__(self,predict): self.predict,self.full,self.grad=predict,0,0
    def __call__(self,x): self.full+=1; return np.asarray(self.predict(np.asarray(x,float).reshape(1,-1))[0],float)

def z(x):
    x1,x2,x3=np.asarray(x,float); return np.array([1,x1,x2,x3,x1*x1,x2*x2,x3*x3,x1*x2,x1*x3,x2*x3])
def dz(x):
    x1,x2,x3=np.asarray(x,float); return np.array([[0,0,0],[1,0,0],[0,1,0],[0,0,1],[2*x1,0,0],[0,2*x2,0],[0,0,2*x3],[x2,x1,0],[x3,0,x1],[0,x3,x2]])

def individual_payoff(B):
    m=B.shape[1]; Xs=[]; cols=[]; evaluation_count=0; gradient_count=0
    for j in range(m):
        candidates=[]
        starts=[np.zeros(3),*list(np.eye(3)*(.99*ALPHA)),*list(-np.eye(3)*(.99*ALPHA))]
        for x0 in starts:
            def objective(x):
                nonlocal evaluation_count; evaluation_count+=1; return float(z(x)@B[:,j])
            def gradient(x):
                nonlocal gradient_count; gradient_count+=1; return dz(x).T@B[:,j]
            r=minimize(objective,x0,jac=gradient,method='SLSQP',bounds=[(-ALPHA,ALPHA)]*3,constraints={'type':'ineq','fun':lambda x:ALPHA**2-x@x},options={'ftol':1e-11,'maxiter':500})
            candidates.append(r)
        valid=[r for r in candidates if r.success and r.x@r.x<=ALPHA**2+1e-7]
        assert valid, f'Nenhum ótimo individual válido para objetivo {j}'
        best=min(valid,key=lambda r:float(r.fun))
        Xs.append(best.x); cols.append(z(best.x)@B)
    # Public orientation: rows objectives, columns individual configurations.
    P=np.column_stack(cols)
    individual_payoff.last_evaluations=evaluation_count; individual_payoff.last_gradient_evaluations=gradient_count
    return np.asarray(Xs),P

def parallel_analysis(P,Xstar,mse,XtX_inv,nmc=2000,seed=777):
    ideal=P.min(1); nadir=P.max(1); amp=np.where(nadir-ideal>1e-12,nadir-ideal,1); Ps=(P-ideal[:,None])/amp[:,None]
    # v0 adapter: internal anchors-in-rows equals public payoff transpose.
    A=Ps.T; assert np.allclose(A.T,Ps)
    E=A[1:]-A[:1]; s=np.linalg.svd(E,compute_uv=False); h=np.einsum('ij,jk,ik->i',np.vstack([z(x) for x in Xstar]),XtX_inv,np.vstack([z(x) for x in Xstar]))
    sd=np.sqrt(np.outer(h,mse))/amp[None,:]; rng=np.random.default_rng(seed); sn=np.empty((nmc,len(s)))
    for b in range(nmc):
        R=rng.normal(0,sd); sn[b]=np.linalg.svd(R[1:]-R[:1],compute_uv=False)
    p95=np.percentile(sn,95,axis=0); d=max(1,int(np.sum(s>p95))); floor=float(s[d]) if d<len(s) else 0.; ceiling=float(s[0]/s[d-1])
    return {'d':d,'s':s,'p95':p95,'floor':floor,'ceiling':ceiling,'scaled':Ps}

def simplex_weights(k,delta):
    p=round(1/delta); return np.array([q for q in product(range(p+1),repeat=k) if sum(q)==p],float)/p

def _nearest_weight_order(weights):
    weights=np.asarray(weights,float); remaining=list(range(len(weights))); order=[remaining.pop(0)]
    while remaining:
        last=weights[order[-1]]; pick=min(remaining,key=lambda i:(float(np.linalg.norm(weights[i]-last)),i)); remaining.remove(pick); order.append(pick)
    return [(i,weights[i]) for i in order]

def _nearest_weight_order(weights):
    weights=np.asarray(weights,float); remaining=list(range(len(weights))); order=[remaining.pop(0)]
    while remaining:
        last=weights[order[-1]]; pick=min(remaining,key=lambda i:(float(np.linalg.norm(weights[i]-last)),i)); remaining.remove(pick); order.append(pick)
    return [(i,weights[i]) for i in order]

def _solve_nbi_base(Bmodel, output_B, indices, Xstar, P, delta, combo_id=0):
    """Único solver-base NBI: equação, jacobianas, starts, resgates e seleção comuns."""
    indices=np.asarray(indices,int); k=len(indices); ideal=P.min(1); amp=np.maximum(P.max(1)-ideal,1e-12); A=((P-ideal[:,None])/amp[:,None]).T
    E=A[1:]-A[:1]; normal=null_space(E).ravel(); normal/=np.linalg.norm(normal)
    if normal@(-A.mean(0))<0: normal=-normal
    rows=[]; full=0; grad=0; warm=None
    for execution_order,(beta_id,beta) in enumerate(_nearest_weight_order(simplex_weights(k,delta))):
        phi=beta@A; vertex=np.flatnonzero(np.isclose(beta,1.,atol=1e-12)&np.isclose(beta.sum(),1.,atol=1e-12))
        if len(vertex)==1:
            x=Xstar[int(vertex[0])].copy(); Fm=z(x)@Bmodel; full+=1; residual=(Fm-ideal)/amp-phi
            rows.append({'k':k,'combo':tuple(indices),'beta_id':beta_id,'beta':beta.copy(),'execution_order':execution_order,'success':bool(np.max(np.abs(residual))<=1e-5),'accepted':bool(np.max(np.abs(residual))<=1e-5),'solver_success':True,'subproblem_status':'COMPLETED','x':x,'F_rsm':z(x)@output_B,'eq_inf':float(np.max(np.abs(residual))),'sphere_violation':max(0.,float(x@x-ALPHA**2)),'start':'payoff_anchor_exact','attempts':0,'attempt_log':[]})
            warm=np.r_[x,0.]; continue
        def predict(x):
            nonlocal full; full+=1; return (z(x)@Bmodel-ideal)/amp
        def jacobian(x):
            nonlocal grad; grad+=1; return (dz(x).T@Bmodel).T/amp[:,None]
        def eq(v): return predict(v[:3])-(phi+v[-1]*normal)
        def jeq(v): return np.column_stack([jacobian(v[:3]),-normal])
        def sphere(v): return ALPHA**2-v[:3]@v[:3]
        def jsphere(v): return np.r_[-2*v[:3],0.]
        def project_t(x):
            nonlocal full; full+=1; return float(((z(x)@Bmodel-ideal)/amp-phi)@normal)
        starts=[]; seen=set()
        def add(name,x,t=None):
            x=np.asarray(x,float); norm=np.linalg.norm(x)
            if norm>ALPHA*(1+1e-10): return
            if norm>ALPHA: x=x*(ALPHA*(1-1e-12)/norm)
            v=np.r_[x,np.clip(project_t(x) if t is None else t,-10,10)]; key=tuple(np.round(v,12))
            if key not in seen: seen.add(key); starts.append((name,v))
        if warm is not None: add('warm',warm[:3],warm[-1])
        add('barycentric_anchor',beta@Xstar)
        for j in np.argsort(-beta): add(f'anchor_{int(indices[j])}',Xstar[j])
        add('center',np.zeros(3)); candidates=[]; attempt_log=[]
        def solve(name,v0,attempt):
            r=minimize(lambda v:-v[-1],v0,jac=lambda v:np.r_[np.zeros(3),-1.],method='SLSQP',bounds=[(-ALPHA,ALPHA)]*3+[(-10,10)],constraints=[{'type':'eq','fun':eq,'jac':jeq},{'type':'ineq','fun':sphere,'jac':jsphere}],options={'ftol':1e-10,'maxiter':500,'disp':False})
            residual=eq(r.x); eq_inf=float(np.max(np.abs(residual))); violation=max(0.,float(r.x[:3]@r.x[:3]-ALPHA**2)); feasible=eq_inf<=1e-5 and violation<=1e-8
            candidates.append((bool(r.success),feasible,r,name,attempt,eq_inf,violation)); attempt_log.append({'attempt':attempt,'start':name,'solver_success':bool(r.success),'eq_inf':eq_inf,'sphere_violation':violation,'t':float(r.x[-1])})
            return candidates[-1]
        attempt=0
        for name,v0 in starts:
            attempt+=1; candidate=solve(name,v0,attempt)
            if candidate[0] and candidate[1]: break
        if not any(c[0] and c[1] for c in candidates):
            rng=np.random.default_rng(1000003+1009*combo_id+beta_id)
            for rescue in range(8):
                direction=rng.normal(size=3); direction/=np.linalg.norm(direction); x=ALPHA*(rng.random()**(1/3))*direction; attempt+=1; candidate=solve(f'rescue_{rescue+1}',np.r_[x,np.clip(project_t(x),-10,10)],attempt)
                if candidate[0] and candidate[1]: break
        chosen=min(candidates,key=lambda c:(0 if c[0] and c[1] else 1,-float(c[2].x[-1]) if c[0] and c[1] else c[5]/1e-5+c[6]/1e-8,c[5],c[6])); solver_success,feasible,r,name,attempt,eq_inf,violation=chosen; x=r.x[:3]
        valid_candidate=bool(solver_success and feasible); subproblem_status='COMPLETED' if valid_candidate else 'NO_FEASIBLE_INTERSECTION'
        rows.append({'k':k,'combo':tuple(indices),'beta_id':beta_id,'beta':beta.copy(),'execution_order':execution_order,'success':valid_candidate,'accepted':valid_candidate,'solver_success':bool(solver_success),'subproblem_status':subproblem_status,'x':x,'F_rsm':z(x)@output_B,'eq_inf':eq_inf,'sphere_violation':violation,'t':float(r.x[-1]),'start':name,'attempts':attempt,'attempt_log':attempt_log})
        warm=r.x.copy() if valid_candidate else None
    return rows,full,grad

def cnbi(B,P,Xstar,mse,XtX_inv):
    pa=parallel_analysis(P,Xstar,mse,XtX_inv,nmc=2000,seed=777); m=B.shape[1]; results=[]; full=0; grad=0; combo_id=0
    for k in range(2,min(m,4)+1):
      for combo in combinations(range(m),k):
        combo=np.asarray(combo,int); A=pa['scaled'][np.ix_(combo,combo)].T; singular=np.linalg.svd(A[1:]-A[:1],compute_uv=False); quality=np.inf if singular[-1]<=1e-12 else singular[0]/singular[-1]
        if not(singular[-1]>pa['floor'] and quality<=pa['ceiling']): continue
        rows,nfull,ngrad=_solve_nbi_base(B[:,combo],B,combo,Xstar[combo],P[np.ix_(combo,combo)],DELTA_BY_K[k],combo_id); results.extend(rows); full+=nfull; grad+=ngrad; combo_id+=1
    pa['rsm_evaluations']=full; pa['gradient_evaluations']=grad
    return results,pa

# SMOKE checks the legacy core without running expensive optimizer comparisons.
diag=pd.read_csv(OUT/'scenario_diagnostics.csv'); rec=diag.iloc[0]; A=np.load(OUT/f"{rec.scenario}_scenario.npz")['anchors']
X=np.vstack([np.array(list(product([-1.,1.],repeat=3))),np.vstack([np.eye(3)*ALPHA,-np.eye(3)*ALPHA]),np.zeros((5,3))]); D=np.vstack([z(x) for x in X]); F=np.sum((X[:,None,:]-A[None,:,:])**2,axis=2)
rng=np.random.default_rng(101); sig=np.sqrt(F.var(0,ddof=1)*(.05/.95)); Y=F+rng.normal(0,sig,F.shape); B=np.linalg.lstsq(D,Y,rcond=None)[0]; E=Y-D@B; mse=np.sum(E*E,axis=0)/(19-10)
Xs,P=individual_payoff(B); pa=parallel_analysis(P,Xs,mse,np.linalg.inv(D.T@D),nmc=2000,seed=777)
raw_smoke,pa_cnbi_smoke=cnbi(B,P,Xs,mse,np.linalg.inv(D.T@D)); assert raw_smoke and all(r['sphere_violation']<=1e-8 for r in raw_smoke); assert any(r['start']=='payoff_anchor_exact' for r in raw_smoke); assert pa_cnbi_smoke['rsm_evaluations']>0 and pa_cnbi_smoke['gradient_evaluations']>0
assert pa['d']>=1 and np.all(np.isfinite(pa['p95'])) and DELTA_BY_K=={2:.1,3:.1,4:.2,5:.5}
pd.DataFrame([{'scenario':rec.scenario,'parallel_d':pa['d'],'svd_calls':2001,'rsm_evaluations_payoff':'instrumented_in_FULL'}]).to_csv(TAB/'smoke_pipeline.csv',index=False)
print('SMOKE pipeline OK; otimização final executada?',CFG['run_optimizers'])


## Execução permanente dos métodos determinísticos

Executa NBI direto, CNBI e VRF-NBI sobre o mesmo RSM pareado e grava checkpoints por método, cenário e semente.

In [ ]:
# PERMANENT PILOT METHOD EXECUTION
# This cell executes deterministic methods and creates one checkpoint per method/scenario/seed.
from factor_analyzer import FactorAnalyzer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

def direct_nbi(B,delta=.20):
    m=B.shape[1]
    if m!=4: return [],{'status':'STRUCTURALLY_INVALID','reason':'m > n_x + 1','expected_subproblems':0,'processed_subproblems':0}
    Xs,P=individual_payoff(B); pe=individual_payoff.last_evaluations; pg=individual_payoff.last_gradient_evaluations
    rows,full,grad=_solve_nbi_base(B,B,np.arange(m),Xs,P,delta)
    failures=sum(not r['success'] for r in rows); status='COMPLETED' if failures==0 else 'COMPLETED_WITH_INFEASIBLE_SUBPROBLEMS'
    return rows,{'status':status,'expected_subproblems':len(rows),'processed_subproblems':len(rows),'converged_fraction':1-failures/max(len(rows),1),'rsm_evaluations':full+pe,'payoff_evaluations':pe,'gradient_evaluations':grad+pg,'payoff_included':True}

def orient_factors(loadings,scores):
    loadings=np.asarray(loadings,float).copy(); scores=np.asarray(scores,float).copy(); dominant=[]; signs=[]
    for q in range(loadings.shape[1]):
        j=int(np.argmax(np.abs(loadings[:,q]))); sign=1. if loadings[j,q]>=0 else -1.; loadings[:,q]*=sign; scores[:,q]*=sign; dominant.append(j); signs.append(sign)
    return loadings,scores,np.asarray(dominant),np.asarray(signs)

def vrf_nbi(B,Yobs,delta=.10):
    scaler=StandardScaler(); Ys=scaler.fit_transform(Yobs); pca=PCA().fit(Ys); cumulative=np.cumsum(pca.explained_variance_ratio_); k=int(np.searchsorted(cumulative,.90)+1)
    if not 2<=k<=4: return [],{'status':'STRUCTURALLY_INVALID','n_factors':k,'reason':'NBI fatorial requer 2 <= fatores <= n_x+1'}
    fa=FactorAnalyzer(n_factors=k,rotation='varimax',method='principal'); scores_raw=fa.fit_transform(Ys); loadings_raw=fa.loadings_.copy(); loadings,scores,dominant,signs=orient_factors(loadings_raw,scores_raw)
    # Invariância exata da orientação diante de troca artificial de sinal.
    flipped_loadings=loadings_raw.copy(); flipped_scores=scores_raw.copy(); flipped_loadings[:,0]*=-1; flipped_scores[:,0]*=-1
    check_loadings,check_scores,_,_=orient_factors(flipped_loadings,flipped_scores); assert np.allclose(check_loadings,loadings) and np.allclose(check_scores,scores)
    Bf=np.linalg.lstsq(D,scores,rcond=None)[0]; Xs,P=individual_payoff(Bf); pe=individual_payoff.last_evaluations; pg=individual_payoff.last_gradient_evaluations
    rows,full,grad=_solve_nbi_base(Bf,B,np.arange(k),Xs,P,delta); failures=sum(not r['success'] for r in rows); status='COMPLETED' if failures==0 else 'COMPLETED_WITH_INFEASIBLE_SUBPROBLEMS'
    return rows,{'status':status,'expected_subproblems':len(rows),'processed_subproblems':len(rows),'converged_fraction':1-failures/max(len(rows),1),'n_factors':k,'pca_cumulative':float(cumulative[k-1]),'rsm_evaluations':full+pe,'payoff_evaluations':pe,'gradient_evaluations':grad+pg,'payoff_included':True,'loadings_original':loadings_raw.tolist(),'loadings_oriented':loadings.tolist(),'dominant_response':dominant.tolist(),'dominant_loading':[float(loadings[j,q]) for q,j in enumerate(dominant)],'sign_applied':signs.tolist(),'scores_before':scores_raw.tolist(),'scores_after':scores.tolist(),'sign_invariance_verified':True}

def save_front(path,rows,meta):
    path.parent.mkdir(parents=True,exist_ok=True)
    Xout=np.vstack([r['x'] for r in rows]) if rows else np.empty((0,3)); Fout=np.vstack([r['F_rsm'] for r in rows]) if rows else np.empty((0,0))
    extras={}
    if rows and 'eq_inf' in rows[0]:
        extras={'k':np.array([r['k'] for r in rows],int),'beta_id':np.array([r['beta_id'] for r in rows],int),'combo_json':np.array([json.dumps(np.asarray(r['combo'],int).tolist()) for r in rows]),'beta_json':np.array([json.dumps(np.asarray(r['beta']).tolist()) for r in rows]),'eq_inf':np.array([r['eq_inf'] for r in rows],float),'sphere_violation':np.array([r['sphere_violation'] for r in rows],float),'start':np.array([r['start'] for r in rows]),'attempts':np.array([r['attempts'] for r in rows],int),'execution_order':np.array([r.get('execution_order',i) for i,r in enumerate(rows)],int),'accepted':np.array([r.get('accepted',r['success']) for r in rows],bool),'solver_success':np.array([r.get('solver_success',r['success']) for r in rows],bool),'subproblem_status':np.array([r.get('subproblem_status','COMPLETED' if r['success'] else 'NO_FEASIBLE_INTERSECTION') for r in rows]),'t':np.array([r.get('t',0.) for r in rows],float),'attempt_log_json':np.array([json.dumps(r.get('attempt_log',[])) for r in rows])}
    np.savez_compressed(path,X=Xout,F_rsm=Fout,success=np.array([r['success'] for r in rows],bool),metadata=json.dumps(meta),**extras)

def execute_deterministic_campaign():
    if not CFG['run_optimizers']: return pd.DataFrame()
    manifest=[]; diag=pd.read_csv(OUT/'scenario_diagnostics.csv')
    for scenario in diag.scenario:
      if scenario not in [f'm{m}_{level}' for m in CFG['scenario_objectives'] for level in CFG['correlation_targets']]: continue
      Atrue=np.load(OUT/f'{scenario}_scenario.npz')['anchors']; m=len(Atrue)
      for seed in CFG['final_seeds']:
        Ftrue=np.sum((X[:,None,:]-Atrue[None,:,:])**2,axis=2); rng=np.random.default_rng(seed); sigma=np.sqrt(Ftrue.var(0,ddof=1)*(.05/.95)); Y=Ftrue+rng.normal(0,sigma,Ftrue.shape)
        B=np.linalg.lstsq(D,Y,rcond=None)[0]; E=Y-D@B; mse=np.sum(E*E,axis=0)/9; Xs,P=individual_payoff(B); payoff_evals=individual_payoff.last_evaluations; payoff_grads=individual_payoff.last_gradient_evaluations
        for method in ['NBI','CNBI','VRF-NBI']:
          identity=checkpoint_identity(scenario,method,seed,B,Atrue); ck=CK/f'{MODE.lower()}_{scenario}_seed{seed}_{method.replace("/","_")}_{identity["fingerprint"][:12]}.npz'; t0=time.perf_counter(); c0=time.process_time(); reused=False
          if ck.exists():
            dat=np.load(ck,allow_pickle=False); meta=json.loads(str(dat['metadata']))
            if checkpoint_matches(meta.get('identity',{}),identity): status=meta['status']; n=len(dat['X']); reused=True
          if not reused:
            if method=='NBI': rows,meta=direct_nbi(B)
            elif method=='VRF-NBI': rows,meta=vrf_nbi(B,Y)
            else:
                rows,pa=cnbi(B,P,Xs,mse,np.linalg.inv(D.T@D)); failures=sum(not r['success'] for r in rows); meta={'status':'COMPLETED' if failures==0 else 'COMPLETED_WITH_INFEASIBLE_SUBPROBLEMS','expected_subproblems':len(rows),'processed_subproblems':len(rows),'infeasible_subproblems':failures,'converged_fraction':1-failures/max(len(rows),1),'parallel_d':pa['d'],'cnbi_subproblem_evaluations':pa['rsm_evaluations'],'payoff_evaluations':payoff_evals,'gradient_evaluations':pa.get('gradient_evaluations',0)+payoff_grads,'rsm_evaluations':pa['rsm_evaluations']+payoff_evals,'payoff_included':True}
            meta['identity']=identity; save_front(ck,rows,meta); status=meta['status']; n=len(rows)
          manifest.append({'scenario':scenario,'seed':seed,'method':method,'status':status,'n_solutions':n,'rsm_evaluations':meta.get('rsm_evaluations',0),'gradient_evaluations':meta.get('gradient_evaluations',0),'converged_fraction':meta.get('converged_fraction',np.nan),'checkpoint_reused':reused,'wall_seconds':time.perf_counter()-t0,'cpu_seconds':time.process_time()-c0,'checkpoint':ck.relative_to(ROOT).as_posix()})
    out=pd.DataFrame(manifest); out.to_csv(TAB/f'{MODE.lower()}_deterministic_runs.csv',index=False); return out

deterministic_manifest=execute_deterministic_campaign()
if CFG['run_optimizers']: print(deterministic_manifest.to_string(index=False))


## Artefatos diagnósticos auditáveis

Exporta RSM, payoff, análise paralela, VRF e o ledger combinação–peso do CNBI, além dos checkpoints agregados retomáveis.

In [ ]:
# SCIENTIFIC DIAGNOSTIC ARTIFACTS
def export_scientific_diagnostics():
    if not CFG['run_optimizers']: return
    rsm_rows=[]; payoff_rows=[]; pa_rows=[]; vrf_rows=[]; ledger=[]; diag=pd.read_csv(OUT/'scenario_diagnostics.csv'); deterministic=pd.read_csv(TAB/f'{MODE.lower()}_deterministic_runs.csv')
    wanted={f'm{m}_{level}' for m in CFG['scenario_objectives'] for level in CFG['correlation_targets']}
    for scenario in diag[diag.scenario.isin(wanted)].scenario:
      Atrue=np.load(OUT/f'{scenario}_scenario.npz')['anchors']; m=len(Atrue)
      for seed in CFG['final_seeds']:
        Ftrue=np.sum((X[:,None,:]-Atrue[None,:,:])**2,axis=2); rng=np.random.default_rng(seed); sigma=np.sqrt(Ftrue.var(0,ddof=1)*(.05/.95)); Y=Ftrue+rng.normal(0,sigma,Ftrue.shape); B=np.linalg.lstsq(D,Y,rcond=None)[0]; fitted=D@B; E=Y-fitted; mse=np.sum(E*E,axis=0)/9; B0=np.linalg.lstsq(D,Ftrue,rcond=None)[0]
        for j in range(m):
          ssr=float(np.sum((Y[:,j]-fitted[:,j])**2)); sst=float(np.sum((Y[:,j]-Y[:,j].mean())**2)); signal=float(np.var(Ftrue[:,j],ddof=1)); noise=float(np.var(Y[:,j]-Ftrue[:,j],ddof=1)); rsm_rows.append({'scenario':scenario,'seed':seed,'objective':j,'n_design':len(D),'design_rank':int(np.linalg.matrix_rank(D)),'r2_target':.95,'r2_observed':1-ssr/sst,'mse_residual':mse[j],'snr_realized':signal/max(noise,1e-15),'noiseless_max_abs_error':float(np.max(np.abs(D@B0[:,j]-Ftrue[:,j])))})
        Xs,P=individual_payoff(B); payoff_rows.append({'scenario':scenario,'seed':seed,'rows':P.shape[0],'columns':P.shape[1],'orientation':'rows_objectives_columns_optima','columns_are_individual_minima':bool(all(np.argmin(P[j])==j for j in range(m))),'diagonal_matches_individual_objective':bool(all(np.isclose(P[j,j],z(Xs[j])@B[:,j]) for j in range(m))),'all_optima_feasible':bool(np.all(np.sum(Xs*Xs,axis=1)<=ALPHA**2+1e-8))})
        pa=parallel_analysis(P,Xs,mse,np.linalg.inv(D.T@D),nmc=2000,seed=777)
        for idx,(sv,p95) in enumerate(zip(pa['s'],pa['p95'])): pa_rows.append({'scenario':scenario,'seed':seed,'singular_index':idx+1,'singular_value':sv,'noise_p95':p95,'retained_d':pa['d'],'n_mc':2000,'mc_seed':777,'mode':'independent_legacy','floor':pa['floor'],'ceiling':pa['ceiling']})
        vrf_path=ROOT/deterministic[(deterministic.scenario==scenario)&(deterministic.seed.astype(int)==int(seed))&(deterministic.method=='VRF-NBI')].checkpoint.iloc[0]
        vrf_data=np.load(vrf_path,allow_pickle=False); vrf_meta=json.loads(str(vrf_data['metadata'])); vrf_data.close(); k=int(vrf_meta['n_factors'])
        vrf_rows.append({'scenario':scenario,'seed':seed,'n_factors':k,'pca_cumulative':float(vrf_meta['pca_cumulative']),'factor_method':'principal','rotation':'varimax','loadings_original_json':json.dumps(vrf_meta['loadings_original']),'loadings_oriented_json':json.dumps(vrf_meta['loadings_oriented']),'dominant_response_json':json.dumps(vrf_meta['dominant_response']),'dominant_loading_json':json.dumps(vrf_meta['dominant_loading']),'sign_applied_json':json.dumps(vrf_meta['sign_applied']),'scores_before_json':json.dumps(vrf_meta['scores_before']),'scores_after_json':json.dumps(vrf_meta['scores_after']),'sign_invariance_verified':bool(vrf_meta['sign_invariance_verified']),'structurally_valid':bool(2<=k<=4)})
        ck=ROOT/deterministic[(deterministic.scenario==scenario)&(deterministic.seed.astype(int)==int(seed))&(deterministic.method=='CNBI')].checkpoint.iloc[0]; dat=np.load(ck,allow_pickle=False); arrays={name:dat[name] for name in dat.files}; dat.close(); successes=np.asarray(arrays['success'],bool); cursor=0
        for cursor in range(len(successes)):
          checkpoint_combo=json.loads(str(arrays['combo_json'][cursor])); checkpoint_beta=json.loads(str(arrays['beta_json'][cursor])); kk=int(arrays['k'][cursor])
          ledger.append({'scenario':scenario,'seed':seed,'combination':json.dumps(checkpoint_combo),'k':kk,'beta_id':int(arrays['beta_id'][cursor]),'beta':json.dumps(checkpoint_beta),'delta':DELTA_BY_K[kk],'execution_order':int(arrays['execution_order'][cursor]),'chosen_start':str(arrays['start'][cursor]),'attempts':int(arrays['attempts'][cursor]),'eq_inf':float(arrays['eq_inf'][cursor]),'sphere_violation':float(arrays['sphere_violation'][cursor]),'t':float(arrays['t'][cursor]),'solver_success':bool(arrays['solver_success'][cursor]),'accepted':bool(arrays['accepted'][cursor]),'success':bool(successes[cursor]),'subproblem_status':str(arrays['subproblem_status'][cursor]),'attempt_log_json':str(arrays['attempt_log_json'][cursor]),'aggregate_checkpoint':ck.relative_to(ROOT).as_posix()})
        assert len({(r['scenario'],r['seed'],r['combination'],r['beta_id']) for r in ledger if r['scenario']==scenario and r['seed']==seed})==len(successes)
    pd.DataFrame(rsm_rows).to_csv(TAB/f'{MODE.lower()}_rsm_diagnostics.csv',index=False); pd.DataFrame(payoff_rows).to_csv(TAB/f'{MODE.lower()}_payoff_diagnostics.csv',index=False); pd.DataFrame(pa_rows).to_csv(TAB/f'{MODE.lower()}_parallel_analysis.csv',index=False); pd.DataFrame(vrf_rows).to_csv(TAB/f'{MODE.lower()}_vrf_diagnostics.csv',index=False); pd.DataFrame(ledger).to_csv(TAB/f'{MODE.lower()}_cnbi_subproblems.csv',index=False)

export_scientific_diagnostics()